## Imports und Pfade

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as sps


PROJECT_DIR = Path.cwd().parent  
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

HARM_DIR = PROCESSED_DIR / "harmonized_250hz"   
SEG_DIR  = PROCESSED_DIR        

SEG_PATH = SEG_DIR / "segments_5s_2p5s_index_clean.parquet"  

OUT_DIR = PROCESSED_DIR / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "features_5s_2p5s_clean_v2_aed.parquet"

print("SEG_PATH exists:", SEG_PATH.exists(), SEG_PATH)
print("HARM_DIR exists:", HARM_DIR.exists(), HARM_DIR)
print("OUT_PATH:", OUT_PATH)

## Bandpass-Filter

In [ ]:
def bandpass_filter(x, fs, low=0.5, high=30.0, order=4):
    x = np.asarray(x, dtype=np.float32)
    nyq = 0.5 * fs
    lowc = low / nyq
    highc = high / nyq
    b, a = sps.butter(order, [lowc, highc], btype="bandpass")
    return sps.filtfilt(b, a, x).astype(np.float32)


## Feature Funktionen

In [ ]:
def calc_vfleak(x):
    x = np.asarray(x, dtype=np.float64).ravel()
    if len(x) < 5:
        return np.nan

    num = np.sum(np.abs(x[1:]))
    den = np.sum(np.abs(x[1:] - x[:-1])) + 1e-12
    N = int(np.floor((np.pi * (num / den)) + 0.5))

    if N <= 0 or N >= len(x) - 1:
        return np.nan

    num2 = np.sum(np.abs(x[N:] + x[:-N]))
    den2 = np.sum(np.abs(x[N:]) + np.abs(x[:-N])) + 1e-12
    return float(num2 / den2)


def calc_xj_x1_x2(x, fs):
    x = np.asarray(x, dtype=np.float64).ravel()
    if len(x) < int(fs):
        return np.nan, np.nan

    slope = np.diff(x)
    slope = np.concatenate([slope, slope[-1:]])
    slope = slope ** 2

    aMs = 0.100
    win = int(round(aMs * fs))
    if win < 1:
        win = 1
    bP = np.ones(win) / win

    tr = int(2 * fs)
    slope_pad = np.concatenate([np.ones(tr) * slope[0], slope, np.ones(tr) * slope[-1]])
    slope_f = sps.lfilter(bP, [1.0], slope_pad)
    slope_f = slope_f[tr: -tr]

    mx = np.max(slope_f) + 1e-12
    x1 = np.percentile(slope_f, 10) / mx

    minAmp = 0.1 * mx
    minDist = int(fs / 10)
    peaks, _ = sps.find_peaks(slope_f, height=minAmp, distance=minDist)
    x2 = float(len(peaks))
    return float(x1), float(x2)


def calc_xi_x3_x4(x, fs, L_sec):
    x = np.asarray(x, dtype=np.float64).ravel()
    N = int(L_sec * fs)
    if len(x) != N:
        x = x[:N] if len(x) > N else np.pad(x, (0, N - len(x)), mode="edge")

    Nfft = 2048
    w = np.hamming(N)
    S = np.fft.fftshift(np.fft.fft(x * w, Nfft))
    ff = np.linspace(-fs / 2, fs / 2, Nfft)
    P = np.abs(S) ** 2

    df = ff[1] - ff[0]
    area = (np.sum(P) * df) / 2.0  
    P = P / (area + 1e-12)

    mask = (ff >= 0) & (ff <= 30)
    f = ff[mask]
    P = P[mask]

    mask_1_10 = (f >= 1) & (f <= 10)
    if not np.any(mask_1_10):
        x3 = np.nan
    else:
        idx = np.argmax(P[mask_1_10])
        x3 = float(f[mask_1_10][idx])

    mask_25_75 = (f >= 2.5) & (f <= 7.5)
    x4 = float(np.sum(P[mask_25_75]) * df) if np.any(mask_25_75) else np.nan
    return x3, x4


def calc_tcsc(x, fs, L_sec):
    x = np.asarray(x, dtype=np.float64).ravel()
    N = int(L_sec * fs)
    if len(x) != N:
        x = x[:N] if len(x) > N else np.pad(x, (0, N - len(x)), mode="edge")

    Ls = 3  
    if L_sec < 3:
        return np.nan

    wls = int(Ls * fs)

    t = np.linspace(0, Ls, wls, endpoint=False)
    w = np.ones_like(t)
    ia = np.where(t < 0.25)[0]
    ib = np.where((t >= 0.25) & (t <= (Ls - 0.25)))[0]
    ic = np.where(t > (Ls - 0.25))[0]
    w[ia] = 0.5 * (1 - np.cos(4 * np.pi * t[ia]))
    w[ib] = 1.0
    w[ic] = 0.5 * (1 - np.cos(4 * np.pi * t[ic]))

    V0 = 0.2
    vals = []
    for j in range(0, int(L_sec - 2)):  
        start = int(j * fs)
        ws = slice(start, start + wls)
        wecg = x[ws] * w

        maxv = np.max(np.abs(wecg)) + 1e-12
        wecg = wecg / maxv

        becg = (np.abs(wecg) > V0).astype(np.float64)
        Nperc = (np.sum(becg) * 100.0) / wls
        vals.append(Nperc)

    return float(np.mean(vals)) if len(vals) else np.nan


def calc_bcp(x, fs):
    x = np.asarray(x, dtype=np.float64).ravel()
    if len(x) < int(2 * fs):
        return np.nan

    L_w = int(2 * fs)
    valC = 0.0055

    slope = np.diff(x)
    slope = np.concatenate([slope, slope[-1:]])
    slope = slope ** 2

    n_w = len(slope) // L_w
    if n_w < 1:
        return np.nan

    bcp_vals = []
    for i in range(n_w):
        slp_w = slope[i * L_w:(i + 1) * L_w]
        mx = np.max(slp_w) + 1e-12
        bcp_vals.append(np.sum(slp_w < valC * mx) / L_w)

    return float(np.min(bcp_vals))


def calc_sampen(x, m=2, r_factor=0.2):
    x = np.asarray(x, dtype=np.float64).ravel()
    N = len(x)
    if N < (m + 2):
        return np.nan

    r = r_factor * (np.std(x) + 1e-12)

    def _phi(mm):
        M = np.array([x[i:i + mm] for i in range(N - mm + 1)])
        count = 0
        total = 0
        for i in range(len(M)):
            dist = np.max(np.abs(M - M[i]), axis=1)
            dist = np.delete(dist, i)  
            total += len(dist)
            count += np.sum(dist <= r)
        return count / (total + 1e-12)

    B = _phi(m)
    A = _phi(m + 1)
    return float(np.log(B + 1e-12) - np.log(A + 1e-12))


## Segment -> Features

In [ ]:
FS = 250
WIN_SEC = 5.0

def segment_to_features_v2(seg_ch0, fs=FS, win_sec=WIN_SEC):
    x = bandpass_filter(seg_ch0, fs=fs, low=0.5, high=30.0, order=4)

    x1, x2 = calc_xj_x1_x2(x, fs)
    x3, x4 = calc_xi_x3_x4(x, fs, L_sec=int(win_sec))
    tcsc = calc_tcsc(x, fs, L_sec=int(win_sec))
    vfleak = calc_vfleak(x)
    bcp = calc_bcp(x, fs)
    sampen = calc_sampen(x)

    return {
        "bcp": bcp,
        "x1": x1,
        "x2": x2,
        "x3_domfreq_1_10": x3,
        "x4_bandpower_2p5_7p5": x4,
        "tcsc": tcsc,
        "vfleak": vfleak,
        "sampen": sampen,
    }


## Segmente laden (pro NPZ featuren)

In [ ]:
seg_index = pd.read_parquet(SEG_PATH)
print("Segments loaded:", len(seg_index))
print(seg_index["label"].value_counts())

def load_npz(npz_file: str):
    d = np.load(HARM_DIR / npz_file, allow_pickle=True)
    sig = d["signal"] 
    fs = int(d["fs"])
    return sig, fs

features_rows = []

for npz_file, grp in seg_index.groupby("npz_file"):
    sig, fs = load_npz(npz_file)
    assert fs == FS, f"{npz_file}: fs={fs} != {FS}"

    for _, row in grp.iterrows():
        s = int(row.start_idx)
        e = int(row.end_idx)
        seg = sig[s:e, 0]  

        feats = segment_to_features_v2(seg, fs=fs, win_sec=WIN_SEC)

        out = {
            "source": row.source,
            "record": row.record,
            "start_s": float(row.start_s),
            "label": int(row.label),
            "npz_file": npz_file,
        }
        out.update(feats)
        features_rows.append(out)

print("Done. Feature rows:", len(features_rows))


## Speichern

In [ ]:
features_df = pd.DataFrame(features_rows)
features_df.to_parquet(OUT_PATH, index=False)
print("Saved:", OUT_PATH.resolve())
